## GX_06 — Customer Segments: Which Groups Should the Bank Treat Differently?

**Question:** *What kinds of customers does the bank have, which of them leave, and what should it offer each group?*

**Notebook Overview**
The churn model from GX_04 gives every customer a risk score: it says who is likely to leave. It does not say what kind of customer that is, so it cannot say which offer to make. A 35-year-old with €120,000 on deposit and a 62-year-old with one product need different conversations, even at the same risk.

Segmentation answers that. We let Spark group customers who look alike, without telling it who left (unsupervised learning, MLlib `KMeans`). Only after the groups exist do we look at how many in each group left.

**Plan:**
1. **Choose** the customer characteristics that describe a customer's relationship with the bank.
2. **Choose** the number of segments, with measures of fit and with business sense.
3. **Describe and name** each segment, and look inside the risky ones (country, age, activity).
4. **Check** that the segments hold for customers they were not built on (validation split).
5. **Combine** segments with the churn model: where does the contact list go, and where does the model miss leavers?
6. **Save** the segmentation model and every customer's segment, for the graph and streaming notebooks.
7. **Recommend** one action per segment.

**What this notebook uses and produces**
| | |
|---|---|
| **Reads** | `gold_churn_features` (7,000 training customers build the segments; 1,499 validation customers check them) and the churn model saved by GX_04 (`/Volumes/workspace/churn/models/churn_final_model`) |
| **Writes** | Table `customer_segments` (all 10,000 customers: id, split, segment) and the segmentation model + its settings in the `models` volume |

**Rules**
- Built on the training customers only**, like the model. Validation customers are only used to check.
- `Exited` is never an input. Segments must exist for a new customer whose future is unknown; the churn rate is only used to *describe* a segment afterwards.
- Test customers get a segment, but their churn is never looked at here. The test set stays reserved for GX_05.

### 0. Preparing the notebook

**Approach:** load the shared helper code. The segmentation helpers live in `src/segments.py`, so the streaming notebook can later put a new customer into a segment with exactly the same code.

**Two settings, and why:**
- `SEEDS = 1…10`: KMeans starts from random points and can end in a different (worse) answer depending on the start. We try 10 starts and keep the best one.
- `K_RANGE = 2…8`: the numbers of segments we compare. More than 8 groups would be too many for a retention team to act on.
- `CHOSEN_K = 5`: the number of segments we keep. It is set here so every later cell uses the same value; Step 3 shows the evidence behind it.

In [0]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import json

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

from src.config import (
    CHURN_MODEL_PATH, CHURN_MODEL_SETTINGS, CUSTOMER_SEGMENTS, GOLD_CHURN,
    MODELS_VOLUME, SEGMENT_MODEL_PATH, SEGMENT_MODEL_SETTINGS,
)
from src.ingest import write_table
from src.ml import KEY_COL, LABEL_COL, SPLIT_COL
from src.segments import (
    CLUSTER_COL, OLDER_SEGMENT_NAME, SEGMENT_FEATURES, SEGMENT_ID_COL, SEGMENT_NAME_COL,
    SEGMENT_VECTOR_COL, add_segment_labels, build_segment_prep_stages, fit_best_kmeans, fixed_layout,
    segment_lookup, segment_profile,
)

SEEDS = list(range(1, 11))   # 10 starting points per k; the lowest-cost result is kept
K_RANGE = range(2, 9)        # numbers of segments tried: 2 to 8
CHOSEN_K = 5                 # the number of segments we use - the decision is explained in Step 3

# Same colours as GX_03, so the slides look like one deck
COLOR_MAIN = "#27ae60"
COLOR_DARK = "#0b5345"
COLOR_GREY = "#7f8c8d"

print("Spark version:", spark.version)
print("Segmentation inputs:", SEGMENT_FEATURES)

### 1. The data, in a fixed layout

**Approach:** read the gold table, keep the 7000 training customers and the 1,499 validation customers. The training customers are sorted by `CustomerId` and kept in one partition (`fixed_layout`).

**Why the fixed layout:** Spark’s KMeans selects its initial cluster centers from a random sample that depends on both the seed and how rows are distributed across partitions. As a result, the same code, seed, and customers can produce different segments on another cluster.

Sorting the rows consistently and using a single partition makes initialization reproducible for the same input. With 7000 rows, this is practical. At millions of rows, we would retain multiple partitions for performance and reuse the saved model to ensure consistent segment assignments (Step 9).

In [0]:
gold = spark.table(GOLD_CHURN)

# Training customers in a fixed order and one partition (see src/segments.fixed_layout):
# this is what makes KMeans give the same segments on every run and every cluster.
train = fixed_layout(gold.filter(F.col(SPLIT_COL) == "train")).cache()
val = gold.filter(F.col(SPLIT_COL) == "val").cache()

for name, df in (("train", train), ("val", val)):
    row = df.agg(F.count("*").alias("n"), F.sum(LABEL_COL).alias("leavers")).first()
    print(f"{name:5s}: {row['n']:,} customers, {row['leavers']:,} leavers ({row['leavers'] / row['n']:.1%})")

**Result:** 7000 training customers, 1,426 leavers (20.4%), and 1499 validation customers, 305 leavers (20.3%) — the split from GX_02, unchanged.

**Problem:** the next cell fits the same KMeans (7 segments, seed 42) on the same training customers spread over 1, 2, 4 and 8 partitions. First in whatever order the data comes, then in the fixed layout. For each run it prints the cost (lower = better segments) and the churn rate of the riskiest segment.

In [0]:
# The same KMeans (7 segments, seed 42) on the same 7,000 customers, only spread over a
# different number of partitions - first as they come, then in the fixed layout.
demo_rows = []
unordered = gold.filter(F.col(SPLIT_COL) == "train")
for layout in ("as the data comes", "fixed layout"):
    for parts in (1, 2, 4, 8):
        df = unordered.repartition(parts)
        if layout == "fixed layout":
            df = fixed_layout(df)
        prepared = Pipeline(stages=build_segment_prep_stages()).fit(df).transform(df)
        model, runs = fit_best_kmeans(prepared, k=7, seeds=[42])
        churn = (model.transform(prepared).groupBy(CLUSTER_COL)
                 .agg(F.round(F.avg(LABEL_COL) * 100, 1).alias("churn")).agg(F.max("churn")).first()[0])
        demo_rows.append({"layout": layout, "partitions": parts,
                          "cost": round(runs[0][1], 1), "riskiest_segment_churn_pct": churn})
display(pd.DataFrame(demo_rows))

**Result:** With the original layout, changing only the partitioning changes the result: both the cost and the riskiest segment vary across runs. With the fixed layout, all four runs produce the same result, with a cost of 9,468.9. However, this single start is substantially worse than the best 7-segment solution found using 10 starts in Step 3, which has a cost of 8,275.3. For this reason, every fit below uses fixed_layout and 10 starts.

**Which characteristics describe a customer?**
KMeans groups customers by distance: customers are considered similar when their feature values are close. Choosing the columns therefore defines what “similar” means. We use four features:

| Column | Why it goes in |
|---|---|
| **Age** | Second-strongest link to leaving in GX_03 (Cramér's V 0.36); also a life stage, which changes the right offer. |
| **Balance** | Money the customer keeps with the bank (V 0.13). A saver and an empty account are different relationships. |
| **NumOfProducts** | Strongest link to leaving (V 0.39), and a lever: the bank can offer a product. |
| **IsActiveMember** | Activity (V 0.17), and a lever: the bank can re-engage a customer. |

**Left out on purpose:**

Exited: excluded for the reasons set out above.

- Tenure, credit score, salary, and credit card ownership: GX_03 found no statistically significant association with leaving (all p > 0.05). They may add noise to the distance calculation; Step 4 tests this rather than assuming it.

- Country and gender: describe who customers are rather than how they use the bank. Gender is also a sensitive attribute. Country is reported by segment in Step 6.

- Engagement and ZeroBalance: derived from the selected columns, so including them would give the same information extra weight.

**First**, we compare stayers and leavers across the four selected features to recap why they matter.

In [0]:
# The same 4 columns, side by side for stayers and leavers (training customers only).
display(
    train.groupBy(LABEL_COL)
    .agg(
        F.count("*").alias("customers"),
        *[F.round(F.avg(c), 2).alias(f"avg_{c}") for c in SEGMENT_FEATURES],
    )
    .orderBy(LABEL_COL)
)

**Result:** Leavers are older (44.9 vs 37.4 years on average), have higher balances (91457€ vs 73337€), and are less likely to be active (35% vs 56%). Their average number of products is similar (1.47 vs 1.54), but this hides the pattern identified in GX_03: churn is lowest among customers with 2 products (7.5%), higher among those with 1 (27.7%), and highest among those with 3–4 (83–100%).

Next, we prepare the features using the training customers. VectorAssembler combines the four columns into a single vector, and StandardScaler standardizes each feature to a mean of 0 and a standard deviation of 1.

**Importance of scaling:** KMeans groups customers using Euclidean distance. Without scaling, a €10000 difference in balance would be numerically 1000 times larger than a 10-year age difference, allowing balance to dominate the segmentation. After scaling, a one-standard-deviation difference in age carries the same weight as a one-standard-deviation difference in balance.

In [0]:
# Preparation (assembler + scaler) is fitted ONCE, on the training customers only.
prep_model = Pipeline(stages=build_segment_prep_stages()).fit(train)
train_prepared = prep_model.transform(train).cache()

scaler = prep_model.stages[-1]
display(pd.DataFrame({
    "column": SEGMENT_FEATURES,
    "mean (learned on train)": [round(float(v), 2) for v in scaler.mean],
    "std (learned on train)": [round(float(v), 2) for v in scaler.std],
}))

**Result:** Using only the training customers, the scaler learned the following means and standard deviations: age 38.94 years (std 10.44), balance €77,028 (std €62,463), and products 1.53 (std 0.58). Mean activity was 0.52, meaning 52% of customers were active.

**How many segments?**
How KMeans works: We choose the number of segments, k. KMeans starts with k centres, assigns each customer to the nearest centre, then moves each centre to the average of its assigned customers. It repeats these steps until the centres stabilize or a stopping limit is reached. Because the result depends on the initial centres, we try 10 starts for each k using fit_best_kmeans and retain the solution with the lowest cost.

**What we measure for each k, from 2 to 8:**
- Cost: the sum of squared distances between customers and their segment centres. Lower is better, but adding segments generally reduces cost. We look for an “elbow”: the point beyond which further reductions become modest.
- Silhouette: calculated with MLlib’s ClusteringEvaluator, this measures how well customers fit within their own segment compared with other segments. Scores range from −1 to 1, higher values indicate clearer separation.
- Starts reaching the best cost: how many of the 10 starts finish within 0.1% of the lowest cost found. A low count suggests that the best solution is easy to miss.
- Smallest segment: its share of all customers. We treat a segment containing only 2% of customers as too small to justify a dedicated campaign.

In [0]:
# Silhouette: how clearly each customer sits in its own segment rather than the nearest
# other one (-1 to 1, higher = clearer). Spark computes it on the whole DataFrame.
evaluator = ClusteringEvaluator(featuresCol=SEGMENT_VECTOR_COL, predictionCol=CLUSTER_COL)
n_train = train.count()

search_rows, best_models = [], {}
for k in K_RANGE:
    model, runs = fit_best_kmeans(train_prepared, k, SEEDS)
    best_models[k] = model
    clustered = model.transform(train_prepared)
    best_cost = min(cost for _, cost in runs)
    smallest = clustered.groupBy(CLUSTER_COL).count().agg(F.min("count")).first()[0]
    search_rows.append({
        "k": k,
        "silhouette": round(evaluator.evaluate(clustered), 3),
        "cost": round(best_cost, 1),
        "starts_reaching_best": sum(cost <= best_cost * 1.001 for _, cost in runs),
        "smallest_segment_pct": round(smallest * 100 / n_train, 1),
    })

k_search = pd.DataFrame(search_rows)
k_search["cost_drop_pct"] = (-k_search["cost"].pct_change() * 100).round(1)
display(k_search)

**Result:**
- Silhouette rises with every extra segment, from 0.407 (k=2) to 0.587 (k=8).
- Cost falls steadily: by 21%, 17%, 17%, 16%, 14% and 8% for each extra segment. There is no sharp elbow.
- Several starts are needed: Only k=2 is found by almost every start (9 of 10). For k=3 to 8, only 1 to 3 of the 10 starts reach the best answer — one single run would usually return a worse segmentation.
- At k=8 the smallest segment is 2.7% of customers. Up to k=7 every segment has at least 8.6%.

The two charts show the same thing. The dashed line marks `CHOSEN_K` (5); the reasons follow below.

In [0]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(k_search["k"], k_search["cost"], marker="o", color=COLOR_MAIN)
ax1.set_title("Cost: total distance to the segment centres (lower = tighter)")
ax1.set_xlabel("Number of segments (k)")
ax1.set_ylabel("Cost")

ax2.plot(k_search["k"], k_search["silhouette"], marker="o", color=COLOR_DARK)
ax2.set_title("Silhouette (higher = clearer separation)")
ax2.set_xlabel("Number of segments (k)")
ax2.set_ylabel("Silhouette")

for ax in (ax1, ax2):
    ax.axvline(CHOSEN_K, color=COLOR_GREY, linestyle="--", linewidth=1)
    ax.set_xticks(list(K_RANGE))
plt.tight_layout()
plt.show()

**Result:** both curves improve smoothly, neither picks a k on its own.

**Decision:**
 - The measures alone cannot decide: Silhouette keeps rising partly for a mechanical reason: activity is a yes/no column and products take only four values, so the data contains blocks of near-identical customers. Every extra segment can split off one more block, which always looks "clearer". That says little about whether the new segment is useful.
- Every one of the five segments is large enoug to act on: the smallest holds 10.1% of customers.
- Five segments can each be described in one line for management (Step 5).
- A 6th segment brings no new kind of customer. To check this, the next cell puts the k=5 and k=6 solutions side by side.

This is the `CHOSEN_K = 5` set in Step 0.

In [0]:
# The k=5 and k=6 solutions side by side: does a 6th segment bring a new kind of customer?
for k in (CHOSEN_K, CHOSEN_K + 1):
    print(f"k = {k}")
    display(
        segment_profile(best_models[k].transform(train_prepared), CLUSTER_COL)
        .select(CLUSTER_COL, "customers", "churn_pct", "avg_age", "avg_balance",
                "one_product_pct", "two_products_pct", "active_pct")
        .orderBy("churn_pct")
    )

**Result:** at k=6 the two active groups stay almost the same (1,213 and 1,771 customers vs 1,259 and 1,764) and the older customers keep their shape (659 instead of 708). The new 6th segment is **inactive customers with savings and 2 or more products** (769 customers), taken mostly from the two inactive groups. It leaves at **31.7%** — almost the same as the inactive savers with exactly 1 product (**32.4%**, 1,387 customers). Both would get the same treatment from the bank. A segment that changes no decision is not worth the extra complexity: **we keep 5.**

### 4. Would more columns give better segments?

The project plan proposed seven clustering columns: tenure, credit score and salary. We left those three out because GX_03 found no link to leaving. Here we test that decision instead of assuming it.

**Approach:** build 5 segments from all seven columns, with exactly the same procedure (training customers, scaling, 10 starts), and look at two things:
- Do the extra columns change the segments: do they differ between segments?
- Are the segments clearer or blurrier (silhouette)?

In [0]:
# Plan's 7 columns = our 4 + the three that showed no link to leaving in GX_03.
PLAN_FEATURES = SEGMENT_FEATURES + ["Tenure", "CreditScore", "EstimatedSalary"]

plan_prep = Pipeline(stages=build_segment_prep_stages(PLAN_FEATURES)).fit(train)
plan_prepared = plan_prep.transform(train).cache()
plan_kmeans, _ = fit_best_kmeans(plan_prepared, CHOSEN_K, SEEDS)
plan_clustered = plan_kmeans.transform(plan_prepared)

display(
    plan_clustered.groupBy(CLUSTER_COL)
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("churn_pct"),
        *[F.round(F.avg(c), 1).alias(f"avg_{c}") for c in PLAN_FEATURES],
    )
    .orderBy("churn_pct")
)
print("Silhouette with 7 columns:", round(evaluator.evaluate(plan_clustered), 3),
      "| with our 4 columns:", k_search.loc[k_search["k"] == CHOSEN_K, "silhouette"].iloc[0])
plan_prepared.unpersist()

**Result:** Adding the three extra features provides no clear benefit.
- The five segments remain almost unchanged: segment sizes differ by at most 7 customers, and churn rates are nearly identical: 10.4%, 12.2%, 21.3%, 27.8%, and 37.3%, compared with 10.5%, 12.3%, 21.2%, 27.8%, and 37.4% using four features.
- The extra features barely vary across segments: average tenure ranges from 4.8 to 5.2 years, credit score from 646 to 656, and salary from €95,742 to €101,665. None clearly distinguishes a segment.
- Silhouette falls to 0.256, compared with 0.492 using four features. This is consistent with adding dimensions that contribute little useful separation. However, silhouette scores calculated on different feature sets are not directly comparable, so this is supporting evidence; the segment sizes, churn rates, and profiles provide the stronger basis for the decision.

**Decision:** Keep the four features. They produce essentially the same segments and are simpler to explain.

### 5. The five segments
**Approach:**

- Combine the fitted preparation steps and the best KMeans model for k = 5 into a single PipelineModel. This assigns customers to segments directly from the raw columns in one call and is the object we save in Step 9.

- Summarize each segment using segment_profile: its size, churn rate, and average feature values.

- Assign descriptive names using segment_name in src/segments.py. Segments with an average age of 50 or above are named “Older customers”. The remaining segments are named by activity level (active / inactive) and balance (savers / little or no savings), based on the majority of customers in each segment.

**Why use a naming rule?** KMeans assigns cluster labels 0–4 in an arbitrary order. Naming segments from their profiles makes the labels meaningful. Separately, segment_id numbers the segments 1–5 by churn rate, from lowest to highest.

In [0]:
# The final segmentation = the fitted preparation + the best k=5 KMeans, as ONE pipeline model.
segment_model = PipelineModel(stages=prep_model.stages + [best_models[CHOSEN_K]])

train_clustered = segment_model.transform(train)
lookup = segment_lookup(segment_profile(train_clustered, CLUSTER_COL).collect())
train_seg = add_segment_labels(train_clustered, lookup).cache()

train_profile = segment_profile(train_seg, SEGMENT_ID_COL, SEGMENT_NAME_COL)
display(train_profile)

**Result:** five clearly different groups.

| # | Segment | Customers | Churn | What defines it |
|---|---|---|---|---|
| 1 | **Active, little or no savings** | 1,259 (18.0%) | **10.5%** | All active; 88% hold no balance; 76% have 2 products; average age 36 |
| 2 | **Active savers** | 1,764 (25.2%) | **12.3%** | All active; average balance €122,619; 70% have 1 product; 36% German |
| 3 | **Inactive, little or no savings** | 1,462 (20.9%) | **21.2%** | None active; 80% hold no balance; 71% have 2 products |
| 4 | **Inactive savers** | 1,807 (25.8%) | **27.8%** | None active; average balance €124,981; 76% have 1 product; 39% German |
| 5 | **Older customers** | 708 (10.1%) | **37.4%** | Average age 60.7, 98% aged 50+; 83% active; balance mixed (31% none) |

**What this shows:**
- Among the younger segments, activity and savings are the main distinctions. Active customers have churn rates of 10–12%, compared with 21–28% for inactive customers. Among inactive customers, those with a balance are more likely to leave.
- Inactive savers combine several warning signs identified in GX_03: inactivity, one product, a positive balance, and a high proportion of German customers. With 502 leavers, this segment accounts for 35% of all 1,426 leavers, making it the largest contributor to churn.
- Could a simple GROUP BY reveal a similar pattern? The four younger segments broadly follow an activity × savings split, so grouping by those features would produce similar groups. KMeans also identifies a distinct older segment, whose youngest customer is about 47. This suggests a useful age boundary, although KMeans does not explicitly select features or learn a cutoff rule. The GROUP BY queries in Step 6 explore these patterns further.
- Customers with 3–4 products do not form a separate segment. They have the highest observed churn rates in GX_03 (83–100%), but represent only 3% of customers and appear across all five segments, accounting for at most 8.6% of any segment. This clustering does not isolate them; the churn model addresses their risk individually.

In [0]:
prof = train_profile.toPandas()   # 5 rows
overall = 100 * train_seg.agg(F.avg(LABEL_COL)).first()[0]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(prof[SEGMENT_NAME_COL], prof["churn_pct"], color=COLOR_MAIN)
for bar, churn, n, share, leavers in zip(bars, prof["churn_pct"], prof["customers"], prof["share_pct"], prof["leavers"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{churn:.1f}%  ({n:,} customers = {share:.0f}%, {leavers} leavers)", va="center", fontsize=9)
ax.axvline(overall, color=COLOR_GREY, linestyle="--", linewidth=1, label=f"All customers ({overall:.1f}%)")
ax.set_xlim(0, prof["churn_pct"].max() * 1.9)
ax.set_xlabel("Churn rate (%)")
ax.set_title("Five customer segments: how many leave")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

### 6. Inside the segments: country and age

The segments were built without country and used age as a continuous feature. We now explore two questions using Spark SQL on the segmented view:

1. Does Germany’s customer mix explain its high churn in GX_03? If the difference mainly reflects a larger share of high-churn segments, Germany’s churn rate should be similar to other countries within each segment.
2. Which older customers are leaving? Most customers in this segment are active, yet its churn rate is 37%.

First, we compare churn rates by segment and country. AVG(CASE WHEN … THEN Exited END) calculates the churn rate for one country: rows from other countries return NULL, which AVG ignores. This lets us show all three countries side by side in a single query.

In [0]:
train_seg.createOrReplaceTempView("segmented")

display(spark.sql(f"""
    SELECT {SEGMENT_ID_COL}, {SEGMENT_NAME_COL},
           ROUND(AVG(CASE WHEN Geography = 'France'  THEN Exited END) * 100, 1) AS france_churn_pct,
           ROUND(AVG(CASE WHEN Geography = 'Germany' THEN Exited END) * 100, 1) AS germany_churn_pct,
           ROUND(AVG(CASE WHEN Geography = 'Spain'   THEN Exited END) * 100, 1) AS spain_churn_pct,
           SUM(CASE WHEN Geography = 'Germany' THEN 1 ELSE 0 END)               AS germany_customers
    FROM segmented
    GROUP BY {SEGMENT_ID_COL}, {SEGMENT_NAME_COL}
    ORDER BY {SEGMENT_ID_COL}
"""))

**Result:** Germany leaves more in every segment. It is not explained by the kind of customers it has.
- Active savers: Germany 17.6% vs France 9.1% and Spain 9.9%.
- Inactive savers: Germany 37.6% vs 20.3% and 24.3%.
- Older customers: Germany 51.2% vs 32.0% and 31.5%.
- German customers are concentrated in the two saver segments (629 + 705 = 1,334 of 1,758, 76%) — Germany has no customers with an empty account (GX_03), so few German customers fall into the "little or no savings" segments.

So, the segment says what offer fits and, the country adds how urgent it is. A German inactive saver is the highest-risk combination below age 50.

**Next steps:** Split the older customers by age band and activity.

In [0]:
display(spark.sql(f"""
    SELECT CASE WHEN Age >= 60 THEN '60+' WHEN Age >= 50 THEN '50-59' ELSE 'under 50' END AS age_band,
           CASE WHEN IsActiveMember = 1 THEN 'active' ELSE 'inactive' END              AS activity,
           COUNT(*)                    AS customers,
           SUM(Exited)                 AS leavers,
           ROUND(AVG(Exited) * 100, 1) AS churn_pct
    FROM segmented
    WHERE {SEGMENT_NAME_COL} = '{OLDER_SEGMENT_NAME}'
    GROUP BY 1, 2
    ORDER BY 1, 2
"""))

**Result:** Among older customers, activity decides almost everything.
- Inactive: 43 of 45 aged 50–59 left (95.6%), and 69 of 78 aged 60+ (88.5%). Only 123 customers, but 112 of the segment's 265 leavers (42%).
- Active 50–59: still 37.6% (287 customers) — being active protects less at this age.
- Active 60+: only 12.8% (281 customers) — the safest group in the segment.
- 17 active customers aged 47–49 also landed here; too few to matter.

**For the bank:** An older customer who becomes inactive is the clearest warning sign in the data.

### 7. Do the segments hold for new customers?

**Approach:** Assign the 1499 validation customers to segments using the fitted model (segment_model.transform), without refitting, exactly as we would for new customers. Then compare segment sizes and churn rates with those in the training data.

**Why:** Useful segments should generalize beyond the 7000 training customers. Similar segment sizes and churn patterns in the validation data support their stability. Large differences or changes in the churn ranking would warrant further investigation, although some variation is expected from a smaller sample.

In [0]:
# Validation customers: put them into the segments learned on train (no refitting).
val_seg = add_segment_labels(segment_model.transform(val), lookup).cache()
val_profile = segment_profile(val_seg, SEGMENT_ID_COL, SEGMENT_NAME_COL)

display(
    train_profile.select(SEGMENT_ID_COL, SEGMENT_NAME_COL,
                         F.col("share_pct").alias("train_share_pct"),
                         F.col("churn_pct").alias("train_churn_pct"))
    .join(val_profile.select(SEGMENT_ID_COL,
                             F.col("customers").alias("val_customers"),
                             F.col("share_pct").alias("val_share_pct"),
                             F.col("churn_pct").alias("val_churn_pct")),
          on=SEGMENT_ID_COL)
    .orderBy(SEGMENT_ID_COL)
)

**Result:** The segments remain broadly stable on validation.
- Segment sizes are very similar: each segment’s share changes by at most 1.3 percentage points (older customers: 10.1% → 11.4%).
- The three highest-risk segments retain their order: older customers (34.5%), inactive savers (27.2%), and inactive customers with little or no savings (18.1%).
- The two active segments have nearly identical validation churn rates: 13.6% and 13.5%, compared with 10.5% and 12.3% in training. With 264 and 370 validation customers, the small training gap may reflect sampling variation. We therefore treat them as one low-risk tier.
- All churn rates remain within about 3 percentage points of their training values.

 ### 8. Segments and the churn model together
The churn model (GX_04) determines who to contact: customers with a predicted churn probability of 0.35 or higher. The segments describe what kind of customer they are. Combining the two answers two questions:
1. Who makes up the contact list? Which segments account for the flagged customers?
2. Where are leavers missed? Segments with many unflagged leavers may need a broader retention approach alongside individual outreach.
Approach: Load the final model and threshold saved in Step 12 of GX_04, score the validation customers, and summarize the results by segment. Run GX_04 first.

**Why use validation customers?** The model was not trained on them, so these are out-of-training predictions. However, GX_04 used these same customers to select the 0.35 threshold. Here, we describe how flagged customers are distributed across segments; we do not treat this as an independent evaluation of model performance.

### 8. Segments and the churn model together

The model (GX_04) decides **who** to contact: every customer with predicted churn probability **≥ 0.35**. The segments say **what kind of customer** each one is. Crossing the two answers two questions:
1. **Where does the contact list go?** Which segments fill it?
2. **Where does the model miss leavers?** A segment whose leavers are rarely flagged needs another approach than a personal call.

**Approach:** load the final model and threshold that GX_04 saved (Step 12 there — run GX_04 first), score the validation customers, and summarise per segment.

**Why validation customers:** the model never trained on them, so their scores are honest. (The 0.35 threshold was chosen on these customers in GX_04; here we only describe *where* flagged customers sit, we do not re-judge the model.)

In [0]:
# The final churn model and its threshold, exactly as GX_04 saved them.
churn_model = PipelineModel.load(CHURN_MODEL_PATH)
with open(CHURN_MODEL_SETTINGS) as f:
    churn_settings = json.load(f)
THRESHOLD = churn_settings["threshold"]
print(f"Churn model: {churn_settings['model']} {churn_settings['layers']}, threshold {THRESHOLD}")

val_scored = (
    churn_model.transform(val_seg)
    .withColumn("p_churn", vector_to_array("probability")[1])
    .withColumn("contact", (F.col("p_churn") >= THRESHOLD).cast("int"))
    .cache()
)

total_contact = val_scored.agg(F.sum("contact")).first()[0]
segment_risk = (
    val_scored.groupBy(SEGMENT_ID_COL, SEGMENT_NAME_COL)
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("actual_churn_pct"),
        F.round(F.avg("p_churn") * 100, 1).alias("avg_predicted_risk_pct"),
        F.sum("contact").alias("contacted"),
        F.round(F.avg("contact") * 100, 1).alias("contacted_pct_of_segment"),
        F.round(F.sum("contact") * 100 / total_contact, 1).alias("share_of_contact_list_pct"),
        F.sum(LABEL_COL).alias("leavers"),
        F.sum(F.col(LABEL_COL) * F.col("contact")).alias("leavers_caught"),
    )
    .withColumn("leavers_caught_pct", F.round(F.col("leavers_caught") * 100 / F.col("leavers"), 1))
    .orderBy(SEGMENT_ID_COL)
)
display(segment_risk)
print(f"Contact list: {total_contact} of {val_scored.count():,} validation customers (P(churn) >= {THRESHOLD})")

**How to read the table** (validation customers, by segment):

- `actual_churn_pct` vs `avg_predicted_risk_pc` : the observed churn rate compared with the model’s average predicted probability of churn.
- `contacted` and `contacted_pct_of_segment` — the number of customers flagged for contact and their share of the segment.
- `share_of_contact_list_pct` — the segment’s share of all customers flagged for contact.
- `leavers_caught` / `leavers` — how many actual leavers were flagged. `leavers_caught_pct` is the model’s recall within that segment.

**Note:** The churn model is a neural network, and Spark’s implementation can produce slightly different results across clusters, even with the same seed (GX_04). The exact counts therefore apply to this run. The cell below prints those counts; the commentary that follows describes only patterns observed across every fit we tried.

**Next**, we chart the number of leavers flagged and missed in each segment.

In [0]:
risk = segment_risk.toPandas()   # 5 rows
missed = risk["leavers"] - risk["leavers_caught"]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(risk[SEGMENT_NAME_COL], risk["leavers_caught"], color=COLOR_DARK, label="Leavers on the contact list (caught)")
ax.barh(risk[SEGMENT_NAME_COL], missed, left=risk["leavers_caught"], color=COLOR_GREY, label="Leavers not on the list (missed)")
for y, (caught, total) in enumerate(zip(risk["leavers_caught"], risk["leavers"])):
    ax.text(total + 1, y, f"{caught}/{total} caught", va="center", fontsize=9)
ax.set_xlim(0, risk["leavers"].max() * 1.3)
ax.set_xlabel("Leavers among the validation customers")
ax.set_title("Where the model catches leavers, and where it misses them")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

# The numbers behind the result text, printed so they always match this run.
for r in risk.itertuples():
    print(f"{getattr(r, SEGMENT_NAME_COL):30s} flagged {r.contacted_pct_of_segment:5.1f}% of the segment, "
          f"{r.share_of_contact_list_pct:5.1f}% of the contact list, caught {r.leavers_caught}/{r.leavers} leavers "
          f"({r.leavers_caught_pct:.1f}%)")

**Result** (exact counts are printed above):

- Inactive savers and older customers account for nearly 6 in 10 customers flagged for contact. Inactive savers alone make up over a third of the list—the largest share of any segment.

- The model catches most leavers in the inactive and older segments: around two thirds or more of each segment’s leavers are flagged.

- Active savers are the model’s main blind spot. It flags only about 1 in 10 customers in this segment and catches roughly 1 in 4 leavers. Among active customers with little or no savings, it catches about half. This suggests that the model’s features are less effective at distinguishing leavers among active savers.
**What this means:** The risk score identifies more leavers in the inactive and older segments. For active savers, outreach based only on the score would miss most leavers. A broad, low-cost action across the segment may therefore be more appropriate (Step 10).

### 9. Saving the segmentation
Approach: Save the segmentation PipelineModel (preparation + KMeans) to the models volume alongside the churn model. Save a small settings file containing the four input columns and the mapping from KMeans cluster labels to segment IDs and names. Then reload the model and verify that every training customer receives the same segment assignment.

**Why save it?**
- The graph notebook (GX_07) compares customer communities with these segments.
- The streaming notebook assigns new customers to segments, allowing alerts to include both their predicted risk and their customer profile to help guide the offer.
- Reproducibility: The saved model defines the segmentation. Later notebooks load it directly, preserving the fitted centres and segment mapping.

In [0]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {MODELS_VOLUME}")
segment_model.write().overwrite().save(SEGMENT_MODEL_PATH)

# What a later notebook needs besides the model: the inputs and the cluster -> segment names.
with open(SEGMENT_MODEL_SETTINGS, "w") as f:
    json.dump({
        "features": SEGMENT_FEATURES,
        "k": CHOSEN_K,
        "segments": {str(c): {"segment_id": sid, "segment_name": name} for c, (sid, name) in lookup.items()},
        "trained_on": "gold_churn_features, split = 'train' (7,000 customers)",
    }, f, indent=2)

# Check: the reloaded model puts every training customer in the same segment as before.
reloaded = PipelineModel.load(SEGMENT_MODEL_PATH)
moved = (
    reloaded.transform(train).select(KEY_COL, F.col(CLUSTER_COL).alias("after"))
    .join(train_seg.select(KEY_COL, F.col(CLUSTER_COL).alias("before")), KEY_COL)
    .filter(F.col("after") != F.col("before"))
    .count()
)
print(f"Saved to {SEGMENT_MODEL_PATH}")
print(f"Training customers in a different segment after reloading: {moved}")
assert moved == 0, "The reloaded model assigns segments differently - do not use it."

**Result:** The model is saved and, reloaded, puts 0 training customers in a different segment.

**Next steps:** Give all 10000 customers their segment and save it as the table `customer_segments` (`CustomerId`, `split`, `segment_id`, `segment_name`), with GX_02's `write_table`.

**Why all 10000:** other notebooks can join on `CustomerId` instead of re-applying the model. Assigning a segment to a test customer uses only their age, balance, products and activity — never their `Exited` value.

In [0]:
# Every customer (train, val and test) gets a segment. No churn label is used here:
# the test customers' Exited values are not read, only their segment is assigned.
customer_segments = (
    add_segment_labels(reloaded.transform(gold), lookup)
    .select(KEY_COL, SPLIT_COL, SEGMENT_ID_COL, SEGMENT_NAME_COL)
)
write_table(customer_segments, CUSTOMER_SEGMENTS)

saved = spark.table(CUSTOMER_SEGMENTS)
print(f"{CUSTOMER_SEGMENTS}: {saved.count():,} customers")
display(
    saved.groupBy(SEGMENT_ID_COL, SEGMENT_NAME_COL)
    .pivot(SPLIT_COL, ["train", "val", "test"])
    .count()
    .orderBy(SEGMENT_ID_COL)
)

**Result:** 10000 customers in `customer_segments`. Each split has a similar mix of segments (for example, older customers: 708 train, 171 validation, 149 test), as expected from the stratified, hash-based split of GX_02.

### 10. One action per segment

Combining the segment profiles (Step 5), the country and age detail (Step 6), the check on new customers (Step 7) and the model view (Step 8):

| Priority | Segment | Share / churn | What we know | Recommended action |
|---|---|---|---|---|
| **1** | **Older customers** | 10% / 37% | 9 in 10 *inactive* older customers leave; active 60+ only 13% | Treat **"older customer becomes inactive"** as an alert: personal call from an advisor within weeks. A dedicated senior advisor for active 50–59 year-olds (still 38%). |
| **1** | **Inactive savers** | 26% / 28%, **35% of all leavers** | Money in the bank, no activity, mostly one product; Germany 38% | **Reactivation offer tied to their savings** (e.g. a savings or investment product as their *second* product). Start with German inactive savers. The model's contact list covers this segment well. |
| **2** | **Inactive, little or no savings** | 21% / 21% | Already 2 products, but inactive | **Low-cost digital re-engagement** (app and e-mail nudges); personal calls only for customers on the model's contact list. |
| **3** | **Active savers** | 25% / 12% | Low churn, but the model catches few of its leavers; Germany 18% vs 9–10% elsewhere | **Broad loyalty action** rather than calls: a second product for the 70% with only one, regular service check-ins. Watch German active savers. |
| **4** | **Active, little or no savings** | 18% / 10.5% | The healthiest group: active, mostly 2 products | **Keep as is**; no retention budget needed. |

**A caution for every offer:** a *second* product protects (7.5% churn), but GX_03 found that customers with 3–4 products leave at 83–100%. Offers should never push customers beyond two products.

11. Summary
We grouped 7000 training customers into five segments using KMeans on age, balance, products, and activity. Features were scaled, and churn was never used as an input. We compared different numbers of segments, tried 10 starts per fit, and checked the results on validation customers. Adding three extra features provided no clear benefit.

**Main findings:**

- Inactive savers and older customers account for 36% of customers but 54% of leavers.

- Churn ranges from 10–12% in the active segments to 37% among older customers. Among older customers, activity makes a large difference.

- Germany has higher churn within every segment, so customer mix alone does not explain the difference.

- The segments remain broadly stable on validation. The two active segments form one low-risk tier.

- The churn model catches most leavers in the inactive and older segments, but misses most among active savers. Broader, low-cost outreach may suit this segment better.
We saved the segmentation model and the customer_segments table for later notebooks. Loading the saved model preserves the segment definitions without retraining.

**Next: GX_07 — Do similar customers leave together?**
GX_07 builds a customer similarity graph using MLlib LSH, then uses GraphFrames to find communities and explore whether customers whose neighbours leave are also more likely to leave. Comparing these communities with our segments will show whether the graph reveals additional patterns.